In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import UnstructuredHTMLLoader
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import os

In [1]:
loader = UnstructuredHTMLLoader(file_path="data/mg-zs-warning-messages.html")
car_docs = loader.load()

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=os.environ["OPENAI_API_KEY"])

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    separators = ["\n"," ", ".", "\n\n"],
    chunk_size = 150,
    chunk_overlap = 100
)
doc = splitter.split_documents(car_docs)

vectorstore = Chroma.from_documents(
    doc,
    embedding = embeddings,
    persist_directory = "path/to/directory"
)

retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k" : 3}
)

answer = """
You can fix your problem by following the guidelines:
guidelines :
{guidelines}

copy:
{copy}

Fixed copy:
"""
prompt_template = ChatPromptTemplate.from_messages([("human" , answer)])

rag_chain = ({ "guidelines" : retriever, "copy": RunnablePassthrough()}
             | prompt_template
             | llm)

answer = rag_chain.invoke("The Gasoline Particular Filter Full warning has appeared. What does this mean and what should I do about it?")

print(answer.content)